In [1]:
import pandas as pd
import numpy as np
import pickle
import glob
import os

import matplotlib.pyplot as plt
%matplotlib inline
import matplotlib
from scipy.stats import gaussian_kde

import matplotlib as mpl
matplotlib.rcParams['font.size'] = 8.0
matplotlib.rcParams['pdf.fonttype'] = 42
mpl.rcParams['font.family'] = 'Arial'

In [2]:
# input all parameters
Fs = 3e4
TimeHistory = 20e-3 # in seconds
TimePost = 2e-3 # in seconds

#for PSTH
bin_size = 0.1
stim_duration = 10.0

In [3]:
filepath_degrees = glob.glob('./dataFolder/4_RawData_Index_Timestamp/ScaledStimulus/*_degrees.pickle')
filepath_notdegrees = glob.glob('./dataFolder/4_RawData_Index_Timestamp/ScaledStimulus/*_scaled.pickle')

out_data = './dataFolder/2_Raster_PSTH/'
out_fig = './dataFolder/Figures/2_Raster_PSTH/'
os.makedirs(out_data, exist_ok=True)
os.makedirs(out_fig, exist_ok=True)

moth_unit = [('m31', 'unit_2')]
             # ('m29', 'unit_1'),
             # ('m30', 'unit_1'),
             # ('m31', 'unit_1'),
             # ('m31', 'unit_2'),
             # # ('m31', 'unit_3'),
             # # ('m11', 'unit_2'),
             # # ('m11', 'unit_3')]

In [5]:
for name, unit_name in moth_unit:
    print(f'processing {name} and {unit_name}')
    
    # get the right timestamp file (degrees if available, else scaled)
    filename = [f for f in filepath_degrees if name in f]
    if not filename:
        filename = [f for f in filepath_notdegrees if name in f][0]
    else:
        filename = filename[0]
    
    with open(filename, "rb") as f:
        testout = pickle.load(f)
    all_Units_aligned = testout[0]
    all_Units_prior = testout[1]
    WhiteNoise = testout[2]
    
    # build PSTH for this unit
    raster = all_Units_aligned[unit_name]
    num_repeats = len(raster)
    spikes = np.concatenate(raster)
    edges = np.arange(0, stim_duration + bin_size, bin_size)
    counts, _ = np.histogram(spikes, bins=edges)
    psth = counts / (num_repeats * bin_size)
    
    # KDE at PSTH bin centers (100 points) and at fine grid (1000 points)
    bin_centers = edges[:-1] + bin_size/2
    kde = gaussian_kde(edges[:-1], weights=psth, bw_method=0.06)
    density_at_psth = kde(bin_centers)
    
    x_grid = np.linspace(edges[:-1].min(), edges[-1].max(), 1000)
    density = kde(x_grid)
    
    # save raster (flat list of spike times + repeat index)
    raster_data = []
    for repeat_idx, spike_arr in enumerate(raster):
        for spike_time in np.atleast_1d(np.array(spike_arr).flatten()):
            raster_data.append({'repeat': repeat_idx, 'spike_time': spike_time})
    pd.DataFrame(raster_data).to_csv(out_data + name + '_' + unit_name + '_raster.csv', index=False)
    
    # save PSTH + KDE at PSTH bin centers
    pd.DataFrame({'bin_center': bin_centers,
                  'firing_rate': psth,
                  'kde_at_bin': density_at_psth}).to_csv(out_data + name + '_' + unit_name + '_psth.csv', index=False)
    
    # save full KDE
    pd.DataFrame({'time': x_grid,
                  'kde': density}).to_csv(out_data + name + '_' + unit_name + '_kde_full.csv', index=False)
    
    # 3-panel figure
    fig, (ax_stim, ax_raster, ax_psth) = plt.subplots(3, 1, figsize=(5, 3), sharex=True,
                                                       gridspec_kw={'height_ratios': [0.5, 2, 1], 'hspace': 0.15})
    
    ax_stim.plot(np.arange(0, len(WhiteNoise)) * 1/Fs, WhiteNoise, color='black', linewidth=1)
    stim_min = round(WhiteNoise.min(), 1)
    stim_max = round(WhiteNoise.max(), 1)
    ax_stim.set_ylim(-25, 25)
    ax_stim.set_yticks([-25, 0, 25])
    ax_stim.set_ylabel('Proboscis\nDeflection\n(degrees)', fontsize=8)
    ax_stim.tick_params(axis='both', labelsize=7)
    
    ax_raster.eventplot(raster, linelengths=0.5, lineoffsets=1, linewidth=1, color='black')
    ax_raster.set_ylabel('Repeats', fontsize=8)
    ax_raster.tick_params(axis='both', labelsize=7)
    
    ax_psth.bar(edges[:-1], psth, width=bin_size, align='edge', edgecolor='black', color='0.3')
    ax_psth.set_ylabel('Firing rate (spikes/s)', fontsize=8)
    ax_psth.set_xlabel('Time (s)', fontsize=8)
    ax_psth.tick_params(axis='both', labelsize=7)
    ax_psth2 = ax_psth.twinx()
    ax_psth2.plot(x_grid, density, color='blue', linewidth=1.5)
    ax_psth2.set_yticks([])
    
    ax_stim.margins(x=0)
    ax_raster.margins(x=0)
    ax_psth.margins(x=0)
    
    plt.subplots_adjust(left=0.22, top=0.95)
    fig.savefig(out_fig + name + '_' + unit_name + '_combined.pdf', bbox_inches='tight', pad_inches=0.1)
    plt.close(fig)

processing m31 and unit_2


In [ ]:
# for k in all_Units_aligned.keys():
#     f = plt.figure(figsize=(20,10))
#     axes = f.add_axes([0.1, 0.8, 1, 0.2])
#     axes2 = f.add_axes([0.1, 0, 1, 0.75])
    
#     print(len(np.concatenate(all_Units_aligned[k])))

#     #can plot here just for raster plot & stim figure- or run last cell to get combined stim, raster, PSTH plot
    
#     axes.plot(np.arange(0,len(WhiteNoise))*1/Fs, WhiteNoise
#               , color = 'black'
#               , linewidth = 1
#              )
#     axes.set_xticklabels([])
#     axes.set_ylabel('Proboscis Movement (Degrees)', fontsize = 15)
    
#     axes2.eventplot(all_Units_aligned[k]
#                   ,linelengths = 0.5
#                  ,lineoffsets = 1
#                  , linewidth = 1
#                  , color = 'black'
#                    )
#     axes2.set_xlim([0, len(WhiteNoise)*1/Fs])
#     axes.set_xlim([0, len(WhiteNoise)*1/Fs])
#     axes2.set_ylabel('Repeats', fontsize = 15)
#     axes2.set_yticklabels(labels = axes2.get_yticks(), fontsize = 10)
#     axes2.set_xlabel('Time (seconds)', fontsize = 15)
#     axes2.set_xticklabels(labels = axes2.get_xticks(), fontsize = 10)
#     stim_min = round(WhiteNoise.min(), 1)
#     stim_max = round(WhiteNoise.max(), 1)
#     axes.set_yticks([stim_min, 0, stim_max])
#     axes.set_yticklabels([stim_min, 0, stim_max], fontsize=10)
    
#     raster_data = []
#     for repeat_idx, spikes in enumerate(all_Units_aligned[k]):
#         for spike_time in np.atleast_1d(np.array(spikes).flatten()):  
#             raster_data.append({'repeat': repeat_idx + 1, 'spike_time': spike_time})
#     raster_df = pd.DataFrame(raster_data)
#     filename = '/Users/meghnaanand/Desktop/Analysis/' + name + '_' + k + '_newraster.csv'
#     raster_df.to_csv(filename, index=False)
#     filename = '/Users/meghnaanand/Desktop/Analysis/' + name + '_' + k + '_newraster.pdf'


    
#     #plt.savefig(filename, bbox_inches='tight')
#     #plt.show()
#     #plt.close(f)

In [ ]:
# #create the PSTH aligning to the raster plot here 
# unit_name = "unit_2" #loop around all units if i want instead of individually 

# bin_size = 0.1 
# stim_duration = 10.0 #duration of one repeat

# raster = all_Units_aligned[unit_name] # extract all the spike and unit data from the raster
# num_repeats = len(raster) 
# spikes = np.concatenate(raster)
# edges = np.arange(0, stim_duration + bin_size, bin_size)
# counts, _ = np.histogram(spikes, bins=edges)
# psth = counts / (num_repeats * bin_size) #takes the count which is total no. of spikes and converts to firing rate (spikes/second)

# #plot the PSTH with Gaussian KDE on the same axis 
# fig, ax1 = plt.subplots(figsize=(20, 3.75))
# ax2 = ax1.twinx()

# ax1.bar(
#     edges[:-1],
#     psth,
#     width=bin_size,
#     align="edge",
#     edgecolor="black",
#     color="0.3")

# kde = gaussian_kde(edges[:-1], weights=psth, bw_method=0.06)
# x_grid = np.linspace(edges[:-1].min(), edges[-1].max(), 1000)
# density = kde(x_grid)
# ax2.plot(x_grid, density, color="blue", linewidth=2.5)

# ax1.set_xlim(0, stim_duration)
# ax1.set_xlabel("Time (s)", fontsize=15)
# ax1.set_ylabel("Firing rate", fontsize=15)
# ax1.tick_params(axis='both', labelsize=10)

# ax2.set_ylabel("KDE Value", fontsize=15, color="blue")
# ax2.tick_params(axis='y', labelsize=10, labelcolor="blue")

# plt.tight_layout()

# #save_path = ("/Users/meghnaanand/Desktop/Analysis/"+ name + "_" + unit_name + "_PSTH.pdf")
# #plt.savefig(save_path, bbox_inches="tight")
# #plt.show()

# full_data_df = pd.DataFrame({
#     'time': x_grid,
#     'kde': density})
# save_path = (f"/Users/meghnaanand/Desktop/Analysis/"f"{name}_{unit_name}_fullgaussiankde.csv")
# full_data_df.to_csv(save_path, index=False)

# kde_at_bins = kde(edges[:-1]) #matches the kde values aligning bin values as opposed to all points for comparison
# full_data_df = pd.DataFrame({
#     'time': edges[:-1],
#     'psth': psth,
#     'kde': kde_at_bins})
# save_path = (f"/Users/meghnaanand/Desktop/Analysis/"f"{name}_{unit_name}_PSTH_data.csv")
# full_data_df.to_csv(save_path, index=False)

In [ ]:
# #make subplot with shared x axis here for figure
# #adjust fig ratio as wanted for paper

# k = 'unit_2'

# fig, (ax_stim, ax_raster, ax_psth) = plt.subplots(3, 1, figsize=(20, 10), sharex=True, gridspec_kw={'height_ratios': [0.5, 2, 1], 'hspace': 0})

# ax_stim.plot(np.arange(0, len(WhiteNoise)) * 1/Fs, WhiteNoise, color='black', linewidth=1)
# ax_stim.set_ylabel('Proboscis\n Movement\n (Degrees)', fontsize=15)
# ax_stim.spines['bottom'].set_visible(False)
# ax_raster.eventplot(all_Units_aligned[k], linelengths=0.5, lineoffsets=1, linewidth=1, color='black')
# ax_raster.set_ylabel('Repeats', fontsize=15)
# ax_raster.spines['bottom'].set_visible(False)
# ax_psth.bar(edges[:-1], psth, width=bin_size, align='edge', edgecolor='black', color='0.3')
# ax_psth.set_ylabel('Firing rate', fontsize=15)
# ax_psth.set_xlabel('Time (s)', fontsize=15)
# ax_psth2 = ax_psth.twinx()
# ax_psth2.plot(x_grid, density, color='blue', linewidth=2.5)
# ax_psth2.set_yticks([])

# ax_stim.margins(x=0)
# ax_raster.margins(x=0)
# ax_psth.margins(x=0)

# plt.subplots_adjust(left=0.15, top=0.95)

# #plt.savefig(f'/Users/meghnaanand/Desktop/{name}_{k}_combined.pdf', bbox_inches='tight')
# plt.show()

In [ ]:
# index = pd.read_csv('/Users/meghnaanand/Desktop/Analysis/' + name + '_index.csv')
# # index_tp = index * 1/Fs
# # index_tp

# # get the first WN repeat 
# start = index.loc[0, :][0]
# stop = index.loc[0, :][1]
# WhiteNoise = Stim[start:stop]